In [1]:
from kinbert_dataset import BertDataset, BertLabeledDataset
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import math
from functools import reduce

import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from tape import ProteinBertForSequenceClassification, TAPETokenizer
from transformers import TrainingArguments, Trainer
import matplotlib.pyplot as plt

from kinbert_mlm import MLM

from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix
import seaborn as sn
import pandas as pd

from eval_utils import get_num_correct, createConfusionMatrix

In [2]:
print(torch.cuda.device_count())
print(torch.cuda.is_available())
print(torch.version.cuda)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"

1
True
11.6


In [3]:
# for vulcan
MODEL_DIR = "../../model/"
BASE_MODEL = "tapePretrainSep_interaction_base_20e_batchsize_8_lr1e5"
MODEL_TAG = "tapeClassify"
DATA_DIR = "../../data/"
DATA_NAME = "ptm_var.pkl"
DATA_TAG = "interations_e2e_1n1p_classed_random"

N_EPOCH = 30
BATCH_SIZE = 16

LR = "1e3"
lr=1e-3

SAVE_TENSORBOARD = MODEL_TAG+"_"+DATA_TAG+"_"+str(N_EPOCH)+"e" + "_batchsize_" + str(BATCH_SIZE) + "_lr_"+LR
SAVE_NAME = SAVE_TENSORBOARD


# zeroshot/model/tapeClassify_interations_e2e_1n1p_classed_random_30e_batchsize_8_lr_1e3

In [4]:
model = ProteinBertForSequenceClassification.from_pretrained("../../model/tapeClassify_interations_e2e_1n1p_classed_random_30e_batchsize_8_lr_1e3_15aa").to(device)
tokenizer = TAPETokenizer(vocab='iupac')

In [5]:
# dat

In [6]:
dat = pd.read_pickle(os.path.join(DATA_DIR,DATA_NAME))
dat["label"] = 1

In [7]:
dataset = BertLabeledDataset(dat,psite_env = 7,check_psite = False)

In [8]:
dataloader = DataLoader(dataset, 
                      batch_size=BATCH_SIZE, 
                      shuffle=False, 
                      collate_fn=dataset.collate_fn)

In [9]:
len(dataloader)

534

In [10]:
# s = dat.iloc[10627,:]

In [11]:
# s

In [12]:
# assert s["substrate_seq"][s["p_ind"]-1] == s["p_aa"]

In [13]:
running_vloss = 0.0
total_vcorrect = 0
n_vsample = 0
model.train(False) # Don't need to track gradents for validation
logits_preds = []
y_preds = [] # save predction
y_trues = [] # save ground truth
class_probs = []
class_labels = []

for j, vdata in enumerate(tqdm(dataloader), 0):
    batch_tokens, input_mask, targets = vdata['input_ids'], vdata['input_mask'], vdata['label']
    (vloss, metrics),logits = model(input_ids = batch_tokens.to(device), input_mask=input_mask.to(device), targets=targets.to(device))
    # total_vcorrect += get_num_correct(logits.to(device), targets.to(device))
    n_vsample += batch_tokens.shape[0]
    y_pred = (torch.max(torch.exp(logits), 1)[1]).data.cpu().numpy()
    y_preds.extend(y_pred)  # save prediction
    y_true = targets.data.cpu().numpy()
    y_trues.extend(y_true)
    logits_preds.append(logits.data.cpu().numpy())

100%|██████████| 534/534 [00:19<00:00, 26.98it/s]


In [14]:
logits_preds = np.concatenate(logits_preds)

In [15]:
len(logits_preds.tolist())

8530

In [16]:
dat["logits"] = logits_preds.tolist()

In [17]:
dat["y_pred"] = y_preds

In [18]:
dat["y_true"] = y_trues

In [20]:
dat.to_pickle("../../data/tapeClassify_ptmvar_predictions_15aa_15aa.pkl")